# 0. Setup and data

Run this notebook once, from inside the `notebooks/` folder. It installs the
dependencies, downloads each benchmark, assembles the files the loaders expect, and
checks that every dataset loads as the real thing rather than a fallback surrogate.

The five benchmarks are IHDP, Twins, ACIC 2016, LaLonde/NSW, and a semi-synthetic ACS
study. Everything lands under `data/raw/` at the repository root.


## Step 1 - install the Python dependencies

The pinned versions are in `requirements.txt`. ACIC additionally needs R with the
`aciccomp2016` package; that step is separate and described below.


In [ ]:
# -q keeps the output short. Re-run is a no-op once the versions are satisfied.
!pip install -q -r ../requirements.txt


## Step 2 - download and assemble IHDP, Twins, and LaLonde

`data/prepare_data.py` fetches the public raw files and writes the processed versions:

- **IHDP**: downloads the standard 100-replication train and test releases and
  concatenates them into the full 747-unit sample (`ihdp_npci_1-100.merged.npz`).
- **Twins**: downloads the GANITE Twins file, derives one-year mortality for each twin,
  and samples a covariate-dependent treatment with a fixed seed, so the file is exactly
  reproducible (`twins.csv`).
- **LaLonde/NSW**: downloads the Dehejia-Wahba experimental sample (185 treated, 260
  control) from the NBER page and assembles `lalonde.csv`.

The output is numerically identical to the files used in the paper. Each raw download
is cached, so re-running is cheap.


In [ ]:
import subprocess
# Runs all three preparation steps. Pass 'ihdp', 'twins', or 'lalonde' to run one.
subprocess.run(['python', 'data/prepare_data.py', 'all'], cwd='..', check=True)


## Step 3 - generate ACIC 2016 (needs R)

ACIC 2016 is produced by the competition's own R package rather than downloaded.
The install path in that package's README is out of date; install from the tarball's
`2016/` subdirectory instead. In R:

```r
# after cloning github.com/vdorie/aciccomp
R CMD INSTALL aciccomp-master/2016
library(aciccomp2016)
sim <- dgp_2016(input_2016, parameters = 7, random.seed = 1)   # the paper's DGP 7
```

Write four files into `data/raw/acic/`: `x.csv` (one-hot the three factor columns to
82 numeric covariates), `z_dgp7.csv` (treatment), `y_dgp7.csv` (observed outcome), and
`potential_outcomes_dgp7.csv` (`y0, y1, mu0, mu1, e`). The loader reads the potential
outcomes so the ground-truth ATE is the true value, not the confounded difference in
means. Full details are in `DATA.md`.


## Step 4 - fetch ACS

folktables downloads the California 2018 person file on first use and caches it under
`data/2018/`. The semi-synthetic treatment and outcome are generated by the loader
(`data/acs_causal.py`) from 20 demographic covariates.


In [ ]:
from folktables import ACSDataSource
# Downloads ~200 MB the first time; cached afterward. No path flag is needed at run
# time once this cache exists.
ACSDataSource(survey_year='2018', horizon='1-Year', survey='person').get_data(
    states=['CA'], download=True)


## Step 5 - verify every dataset loads as real

`verify_provenance.py` loads each dataset through the same entry point the experiment
driver uses and asserts none falls back to a synthetic surrogate. It prints the source,
sample size `n`, covariate count `d`, and true ATE for each one. Expected true ATEs:
IHDP about 4.0, Twins about -0.016, ACIC about 3.95, LaLonde about 1794 (dollars).


In [ ]:
subprocess.run(['python', 'data/verify_provenance.py'], cwd='..', check=True)
